In [22]:
!nvidia-smi

Sun Apr 12 08:08:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P0             25W /   70W |     547MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [23]:
!pip install git+https://github.com/openai/whisper.git -q
!pip install gradio -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [24]:
!pip install yt-dlp

In [25]:
import whisper
# from whisper.utils import write_vtt

In [26]:
model = whisper.load_model("medium") #diff models - tiny ,small , large , medium ...

In [27]:
link = "https://music.youtube.com/watch?v=L-98EuM8CYM&si=eymcTagyRIhii7-q"

In [28]:
import requests # to get the content of the link
import re # to find its mp3 file

In [29]:
content = requests.get(link)

In [30]:
import yt_dlp

url = "https://music.youtube.com/watch?v=L-98EuM8CYM&si=59RhG2OuUDN5FGcZF"

ydl_opts = {
    'format': 'bestaudio/best',
    'outtmpl': 'podcast.%(ext)s',
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'mp3',
    }],
    # 'cookiesfrombrowser': ('chrome',)
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([url])

[youtube] Extracting URL: https://music.youtube.com/watch?v=L-98EuM8CYM&si=59RhG2OuUDN5FGcZF
[youtube] L-98EuM8CYM: Downloading webpage


[youtube] L-98EuM8CYM: Downloading android vr player API JSON


ERROR: [youtube] L-98EuM8CYM: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


DownloadError: ERROR: [youtube] L-98EuM8CYM: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

In [ ]:
result = model.transcribe('podcast.mp3')

In [ ]:
result

In [ ]:
def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds - int(seconds)) * 1000)
    return f"{hours:02}:{minutes:02}:{secs:02},{millis:03}"


with open("output.srt", "w") as f:
    for i, seg in enumerate(result["segments"]):
        start = format_time(seg['start'])
        end = format_time(seg['end'])
        text = seg['text']

        f.write(f"{i+1}\n")
        f.write(f"{start} --> {end}\n")
        f.write(f"{text}\n\n")

In [ ]:
with open("output.srt", "r") as f:
    print(f.read())

In [ ]:
!pip install --upgrade transformers

In [ ]:
import gradio as gr
import whisper
import yt_dlp
import os

# Load Whisper model
model = whisper.load_model("base")

def inference(audio_file, link):

    audio_path = None

    # 🔹 OPTION 1: Upload audio
    if audio_file is not None:
        audio_path = audio_file

    # 🔹 OPTION 2: YouTube link
    elif link:
        try:
            ydl_opts = {
                'format': 'bestaudio/best',
                'outtmpl': 'podcast.%(ext)s',
                'postprocessors': [{
                    'key': 'FFmpegExtractAudio',
                    'preferredcodec': 'mp3',
                }],
                'quiet': True,
                'noplaylist': True
            }

            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([link])

            # Detect downloaded file
            if os.path.exists("podcast.mp3"):
                audio_path = "podcast.mp3"
            elif os.path.exists("podcast.webm"):
                audio_path = "podcast.webm"

        except Exception as e:
            return f"❌ Download Error: {str(e)}", None

    else:
        return "⚠️ Please upload audio or provide a link", None

    # 🔥 MAIN PART: Translate to English
    result = model.transcribe(
        audio_path,
        task="translate",   # 👈 IMPORTANT
        # fp16=False          # safer for CPU
    )

    # 🔹 Time formatting
    def format_time(seconds):
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        millis = int((seconds - int(seconds)) * 1000)
        return f"{hours:02}:{minutes:02}:{secs:02},{millis:03}"

    # 🔹 Create subtitle file
    srt_file = "subtitles.srt"

    with open(srt_file, "w", encoding="utf-8") as f:
      for i, seg in enumerate(result["segments"]):
          start = format_time(seg['start'])
          end = format_time(seg['end'])
          text = seg['text']

          f.write(f"{i+1}\n{start} --> {end}\n{text}\n\n")

  # 🔥 Read and return file content
    with open(srt_file, "r", encoding="utf-8") as f:
      srt_content = f.read()

    return srt_content, srt_file


# ---------------- UI ---------------- #

block = gr.Blocks()

with block:
    gr.HTML("""
    <center>
        <h1>🎧 PodScript</h1>
        <p>Upload audio OR paste YouTube link → Get English Transcript + Subtitles</p>
    </center>
    """)

    with gr.Group():

        audio = gr.Audio(type="filepath", label="Upload Podcast Audio")

        link = gr.Textbox(label="YouTube Link (Optional)")

        btn = gr.Button("Get PodScript 🪄")

        text = gr.Textbox(
            label="Translated Transcript (English)",
            lines=8
        )

        file_out = gr.File(label="Download Subtitles (SRT)")

        btn.click(
            inference,
            inputs=[audio, link],
            outputs=[text, file_out]
        )

block.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://03f2b4009416943dff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
